<a href="https://colab.research.google.com/github/rcst-thesis/AutoTranslation/blob/main/AutoTranslation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb -q
!google-chrome --version

In [ ]:
!pip install selenium webdriver-manager -q

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import csv, os

filename = list(uploaded.keys())[0]
ext = os.path.splitext(filename)[1].lower()

INPUT_PHRASES = []

if ext == ".txt":
    with open(filename, encoding="utf-8") as f:
        rows_raw = [line.strip() for line in f if line.strip()]
    for line in rows_raw:
        parts = line.split(",", 1)
        if len(parts) == 2:
            INPUT_PHRASES.append((parts[0].strip(), parts[1].strip()))

elif ext == ".csv":
    with open(filename, encoding="utf-8") as f:
        reader = csv.reader(f)
        headers = next(reader, None)
        if headers and not headers[0].strip().isdigit():
            headers = [h.strip().lower() for h in headers]
            eng_idx = next((i for i, h in enumerate(headers) if "english" in h or "eng" in h), 0)
            id_idx  = next((i for i, h in enumerate(headers) if "id" in h), None)
            for row in reader:
                if len(row) > eng_idx:
                    num = row[id_idx].strip() if id_idx is not None else str(len(INPUT_PHRASES) + 1)
                    phrase = row[eng_idx].strip()
                    if phrase:
                        INPUT_PHRASES.append((num, phrase))
        else:
            if headers:
                if len(headers) == 2:
                    INPUT_PHRASES.append((headers[0].strip(), headers[1].strip()))
                elif len(headers) == 1:
                    INPUT_PHRASES.append((str(len(INPUT_PHRASES) + 1), headers[0].strip()))
            for row in reader:
                if len(row) >= 2:
                    INPUT_PHRASES.append((row[0].strip(), row[1].strip()))
                elif len(row) == 1:
                    INPUT_PHRASES.append((str(len(INPUT_PHRASES) + 1), row[0].strip()))

else:
    print(f"Unsupported file type: {ext}. Use .txt or .csv")

print(f"Loaded {len(INPUT_PHRASES)} phrases from {filename}\n")
print("First 5:")
for num, phrase in INPUT_PHRASES[:5]:
    print(f"  [{num}] {phrase}")

print("\nMulti-sentence sample:")
for num, phrase in INPUT_PHRASES:
    if ". " in phrase:
        print(f"  [{num}] {phrase}")
        break

In [ ]:
import csv, time, urllib.parse
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ── Config ────────────────────────────────────────────
SOURCE_LANG = "en"
TARGET_LANG = "hil"
OUTPUT_FILE = "output.csv"
DELAY       = 1.5

# ── Driver ────────────────────────────────────────────
def build_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1280,800")
    opts.add_argument("--remote-debugging-port=9222")
    opts.add_argument("--disable-extensions")
    opts.add_argument("--disable-setuid-sandbox")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts
    )

# ── Translate ─────────────────────────────────────────
def translate(driver, text):
    url = (
        f"https://translate.google.com/?hl=en&sl={SOURCE_LANG}"
        f"&tl={TARGET_LANG}&text={urllib.parse.quote(text)}&op=translate"
    )
    driver.get(url)
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "span[jsname='W297wb']")
            )
        )
        els = driver.find_elements(By.CSS_SELECTOR, "span[jsname='W297wb']")
        result = " ".join(el.text.strip() for el in els if el.text.strip())
    except Exception:
        result = ""
    time.sleep(DELAY)
    return result

# ── Main ──────────────────────────────────────────────
driver = build_driver()
results = []

print(f"Translating {len(INPUT_PHRASES)} phrases [{SOURCE_LANG} → {TARGET_LANG}]...\n")

for i, (num, phrase) in enumerate(INPUT_PHRASES, 1):
    print(f"[{i}/{len(INPUT_PHRASES)}] {phrase}", end=" ... ")
    translation = translate(driver, phrase)
    print(translation)
    results.append({"id": num, "english": phrase, "hiligaynon": translation})

    # Save progress every 50 phrases
    if i % 50 == 0:
        with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["id", "english", "hiligaynon"])
            writer.writeheader()
            writer.writerows(results)
        print(f"  >> Progress saved at {i} phrases.")

driver.quit()

# Final save
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "english", "hiligaynon"])
    writer.writeheader()
    writer.writerows(results)

print(f"\nDone. Saved {len(results)} rows to {OUTPUT_FILE}")

In [ ]:
from google.colab import files
files.download("output.csv")